# 📊 Data Analysis — 10-Week Course
## Week 3: Data Cleaning & Preprocessing

---
### Learning Objectives
- Handle missing values (detect, evaluate, impute, drop)
- Remove and prevent duplicate records
- Fix inconsistent formatting: case, whitespace, date formats, units
- Detect and treat outliers
- Apply normalisation and standardisation
- Build a reusable cleaning pipeline

### Outline
1. Loading Dirty Data  
2. Missing Values  
3. Duplicates  
4. Inconsistent Formatting  
5. Outlier Detection & Treatment  
6. Normalisation & Standardisation  
7. Building a Cleaning Pipeline  
8. Lab Exercises  
9. Assignment  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats.mstats import winsorize

np.random.seed(42)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

---
## 1. Loading Dirty Data

Real-world data is almost never clean. We deliberately introduce common errors into the Africa health dataset to practise fixing them.

In [ ]:
N = 54
countries = [
    "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
    "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
    "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
    "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
    "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
    "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
    "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
    "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
]
regions = (["West Africa"]*16+["East Africa"]*14+
           ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)

df_clean_orig = pd.DataFrame({
    "country"           :countries,
    "region"            :regions,
    "life_expectancy"   :np.clip(np.random.normal(63,8,N),45,80),
    "infant_mortality"  :np.clip(np.random.exponential(35,N),5,120),
    "gdp_per_capita"    :np.clip(np.exp(np.random.normal(7.2,1.2,N)),300,15000),
    "vaccination_dtp3"  :np.clip(np.random.normal(78,18,N),20,99),
    "water_access"      :np.clip(np.random.normal(68,22,N),15,99),
    "health_expenditure":np.clip(np.random.normal(5.2,2.1,N),1,12),
})

# ── Introduce dirty data ─────────────────────────────────────────────
dirty = df_clean_orig.copy()

# 1. Missing values
for col, pct in [("life_expectancy",0.10),("gdp_per_capita",0.15),("vaccination_dtp3",0.08)]:
    idx = np.random.choice(dirty.index, int(pct*N), replace=False)
    dirty.loc[idx, col] = np.nan

# 2. Duplicate rows
dupe_rows = dirty.sample(3, random_state=7)
dirty = pd.concat([dirty, dupe_rows], ignore_index=True)

# 3. Inconsistent formatting
dirty.loc[0,  "country"] = "  nigeria "    # extra whitespace
dirty.loc[1,  "country"] = "ETHIOPIA"       # wrong case
dirty.loc[2,  "region"]  = "west africa"    # inconsistent case
dirty.loc[3,  "region"]  = "NORTH AFRICA"   # inconsistent case

# 4. Outliers (data entry errors)
dirty.loc[10, "life_expectancy"]  = 200    # impossible value
dirty.loc[15, "gdp_per_capita"]   = -500   # negative GDP
dirty.loc[20, "vaccination_dtp3"] = 150    # > 100%

# 5. Wrong data type (stored as string)
dirty["gdp_per_capita"] = dirty["gdp_per_capita"].astype(str)
dirty.loc[5, "gdp_per_capita"] = "N/A"     # mixed type

dirty.to_csv("africa_health_dirty.csv", index=False)
print("Dirty dataset shape:", dirty.shape)
print("Issues introduced: missing values, duplicates, inconsistent formatting, outliers, wrong types")
dirty.head(8)

---
## 2. Missing Values

In [ ]:
df = pd.read_csv("africa_health_dirty.csv")

# Detect missing values
miss = df.isnull().sum()
miss_pct = df.isnull().mean() * 100
miss_df = pd.DataFrame({"count": miss, "pct": miss_pct.round(1)})
miss_df = miss_df[miss_df["count"] > 0]

print("Missing Values:")
print(miss_df.to_string())

In [ ]:
# ── Strategy 1: Drop rows with too many NaNs ─────────────────────────
# Drop rows where > 50% of values are missing
threshold = 0.5 * df.shape[1]
df_dropped = df.dropna(thresh=threshold)
print(f"Rows after dropping (>50% missing): {len(df_dropped)} (removed {len(df)-len(df_dropped)})")

# ── Strategy 2: Impute with mean / median ────────────────────────────
df_imputed = df.copy()

# Force numeric (NaN where conversion fails)
for col in ["life_expectancy","gdp_per_capita","vaccination_dtp3","water_access","health_expenditure"]:
    df_imputed[col] = pd.to_numeric(df_imputed[col], errors="coerce")

# Median imputation for skewed, mean for symmetric
df_imputed["gdp_per_capita"]    = df_imputed["gdp_per_capita"].fillna(df_imputed["gdp_per_capita"].median())
df_imputed["life_expectancy"]   = df_imputed["life_expectancy"].fillna(df_imputed["life_expectancy"].mean())
df_imputed["vaccination_dtp3"]  = df_imputed.groupby("region")["vaccination_dtp3"]\
                                       .transform(lambda x: x.fillna(x.median()))

print(f"Missing values remaining: {df_imputed.isnull().sum().sum()}")

---
## 3. Duplicates

In [ ]:
df_work = df_imputed.copy()

print(f"Total rows before dedup: {len(df_work)}")
print(f"Duplicate rows found: {df_work.duplicated().sum()}")

# Show the duplicates
dupes = df_work[df_work.duplicated(keep=False)]
print("\nDuplicated records (country column):")
print(dupes["country"].tolist())

# Remove duplicates — keep first occurrence
df_work = df_work.drop_duplicates(keep="first").reset_index(drop=True)
print(f"\nRows after dedup: {len(df_work)}")

---
## 4. Inconsistent Formatting

In [ ]:
print("BEFORE cleaning:")
print(df_work[["country","region"]].head(8).to_string())

# Fix whitespace and case
df_work["country"] = df_work["country"].str.strip().str.title()
df_work["region"]  = df_work["region"].str.strip().str.title()

print("\nAFTER cleaning:")
print(df_work[["country","region"]].head(8).to_string())

# Verify unique regions
print("\nUnique regions:", sorted(df_work["region"].unique()))

In [ ]:
# Handling mixed date formats
date_samples = pd.DataFrame({"date_str": [
    "2022-01-15", "15/01/2022", "January 15, 2022",
    "2022.01.15", "01-15-2022", "20220115"
]})

date_samples["parsed_date"] = pd.to_datetime(
    date_samples["date_str"], infer_datetime_format=True, errors="coerce"
)
date_samples["year"]  = date_samples["parsed_date"].dt.year
date_samples["month"] = date_samples["parsed_date"].dt.month
print(date_samples)

---
## 5. Outlier Detection & Treatment

In [ ]:
# Detect impossible values
checks = {
    "life_expectancy"   : (0,   120),
    "gdp_per_capita"    : (0,   100000),
    "vaccination_dtp3"  : (0,   100),
    "water_access"      : (0,   100),
    "health_expenditure": (0,   50),
}

print("IMPOSSIBLE VALUE VIOLATIONS:")
for col, (lo, hi) in checks.items():
    if col not in df_work.columns:
        continue
    bad = df_work[(df_work[col] < lo) | (df_work[col] > hi)]
    if len(bad):
        print(f"  {col:<25}: {len(bad)} violations  Values: {bad[col].tolist()}")
    else:
        print(f"  {col:<25}: OK")

# Fix impossible values → NaN, then impute
for col, (lo, hi) in checks.items():
    if col in df_work.columns:
        df_work.loc[(df_work[col] < lo) | (df_work[col] > hi), col] = np.nan
        df_work[col] = df_work[col].fillna(df_work[col].median())

print("\nAfter fixing impossible values:")
for col, (lo, hi) in checks.items():
    if col in df_work.columns:
        bad = df_work[(df_work[col] < lo) | (df_work[col] > hi)]
        print(f"  {col:<25}: {len(bad)} violations")

In [ ]:
# IQR-based outlier detection and Winsorization
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k*(q3-q1), q3 + k*(q3-q1)

cols_to_check = ["infant_mortality", "gdp_per_capita"]

fig, axes = plt.subplots(len(cols_to_check), 2, figsize=(12, 4*len(cols_to_check)))

for i, col in enumerate(cols_to_check):
    if col not in df_work.columns: continue
    raw   = df_work[col].dropna()
    wins  = np.array(winsorize(raw, limits=[0.05, 0.05]))
    lo, hi = iqr_bounds(raw)
    n_out   = ((raw < lo) | (raw > hi)).sum()

    axes[i,0].boxplot([raw, wins], labels=["Before", "After Winsorize"],
                      patch_artist=True,
                      boxprops=dict(facecolor="#AED6F1"),
                      medianprops=dict(color="navy",linewidth=2),
                      flierprops=dict(marker="o",color="crimson",markersize=5))
    axes[i,0].set_title(f"{col} — Box Plot ({n_out} IQR outliers)")

    axes[i,1].hist(raw,  bins=18, alpha=0.5, label="Before", color="#E74C3C", density=True)
    axes[i,1].hist(wins, bins=18, alpha=0.5, label="After",  color="#27AE60", density=True)
    axes[i,1].set_title(f"{col} — Histogram")
    axes[i,1].legend()

plt.suptitle("Outlier Treatment — Before vs After Winsorization", fontsize=12)
plt.tight_layout()
plt.savefig("week3_outliers.png", bbox_inches="tight")
plt.show()

---
## 6. Normalisation & Standardisation

| Method | Formula | Range | When to use |
|---|---|---|---|
| Min-Max Normalisation | `(x - min) / (max - min)` | [0, 1] | When bounded range needed |
| Z-score Standardisation | `(x - mean) / std` | Unbounded (≈ -3 to 3) | When comparing across features |
| Robust Scaling | `(x - median) / IQR` | Unbounded | When outliers are present |

In [ ]:
numeric_cols = ["life_expectancy","infant_mortality","gdp_per_capita",
                "vaccination_dtp3","water_access","health_expenditure"]
numeric_cols = [c for c in numeric_cols if c in df_work.columns]
data = df_work[numeric_cols].dropna()

# Min-Max Normalisation
df_minmax = (data - data.min()) / (data.max() - data.min())

# Z-score Standardisation
df_zscore = (data - data.mean()) / data.std()

# Robust Scaling
df_robust = (data - data.median()) / (data.quantile(0.75) - data.quantile(0.25))

# Compare ranges
print(f"{'Column':<25} {'Original':>12} {'MinMax':>12} {'ZScore':>12} {'Robust':>12}")
print("-" * 75)
for col in numeric_cols:
    print(f"{col:<25} [{data[col].min():>5.1f},{data[col].max():>6.1f}]"
          f" [{df_minmax[col].min():.2f},{df_minmax[col].max():.2f}]"
          f" [{df_zscore[col].min():.2f},{df_zscore[col].max():.2f}]"
          f" [{df_robust[col].min():.2f},{df_robust[col].max():.2f}]")

---
## 7. Building a Cleaning Pipeline

In [ ]:
def clean_health_data(filepath):
    """
    Full cleaning pipeline for the Africa Health dataset.
    Returns a cleaned DataFrame and a report dictionary.
    """
    report = {}

    # Step 1: Load
    df = pd.read_csv(filepath)
    report["original_shape"] = df.shape

    # Step 2: Fix data types
    numeric = ["life_expectancy","infant_mortality","gdp_per_capita",
                "vaccination_dtp3","water_access","health_expenditure"]
    for col in numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Step 3: Standardise text
    for col in ["country","region"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()

    # Step 4: Remove duplicates
    before = len(df)
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    report["duplicates_removed"] = before - len(df)

    # Step 5: Fix impossible values → NaN
    bounds = {
        "life_expectancy"   : (0,   120),
        "gdp_per_capita"    : (0,   100000),
        "vaccination_dtp3"  : (0,   100),
        "water_access"      : (0,   100),
        "health_expenditure": (0,   50),
    }
    impossible_count = 0
    for col, (lo, hi) in bounds.items():
        if col not in df.columns: continue
        mask = (df[col] < lo) | (df[col] > hi)
        impossible_count += mask.sum()
        df.loc[mask, col] = np.nan
    report["impossible_values_fixed"] = int(impossible_count)

    # Step 6: Impute missing values
    report["missing_before"] = int(df.isnull().sum().sum())
    for col in numeric:
        if col in df.columns:
            if "region" in df.columns:
                df[col] = df.groupby("region")[col].transform(
                    lambda x: x.fillna(x.median())
                )
            df[col] = df[col].fillna(df[col].median())  # fallback
    report["missing_after"] = int(df.isnull().sum().sum())

    report["clean_shape"] = df.shape
    return df, report


# Run the pipeline
df_cleaned, cleaning_report = clean_health_data("africa_health_dirty.csv")

print("CLEANING PIPELINE REPORT")
print("=" * 40)
for k, v in cleaning_report.items():
    print(f"  {k:<30}: {v}")

df_cleaned.to_csv("africa_health_cleaned.csv", index=False)
print("\nCleaned dataset saved to africa_health_cleaned.csv")

---
## 8. Lab Exercises

### Lab 1: Missing Value Strategy Comparison
For `vaccination_dtp3` with 8% missing values, compare three strategies:  
(a) Drop rows, (b) Global median, (c) Region-conditional median.  
Plot the resulting distributions and compute mean/std for each.

In [ ]:
# Lab 1


### Lab 2: Scaling Comparison
Apply min-max, z-score, and robust scaling to `gdp_per_capita`.  
Create a 3-panel figure showing the distribution after each transformation.  
Which transformation makes the data look most normal?

In [ ]:
# Lab 2


---
## 9. Assignment — Week 3

**Problem 1 (30 pts): Dirty Dataset Audit**  
Load `africa_health_dirty.csv`. Write a function `full_audit(df)` that reports:  
- Number and % of missing values per column  
- Number of duplicate rows  
- Any impossible values (define ranges yourself)  
- Type mismatches  

**Problem 2 (40 pts): Clean the Dataset**  
Extend `clean_health_data()` to also:  
- Detect and Winsorize statistical outliers (IQR method, k=1.5)  
- Add a `gdp_band` column (Low/Lower-Mid/Upper-Mid/High using quartiles)  
- Standardise all numeric columns using z-score (store in `*_scaled` columns)  

**Problem 3 (30 pts): Before/After Visualisation**  
Create a 2-row figure comparing the distribution of 3 variables before and after your complete cleaning pipeline. Add a text box to each panel with key statistics (mean, std, n_missing).

In [ ]:
# Problem 1


In [ ]:
# Problem 2


In [ ]:
# Problem 3


---
## Summary — Week 3

| Task | Tool |
|---|---|
| Detect missing | `df.isnull().sum()` |
| Impute global | `fillna(mean/median)` |
| Impute by group | `groupby().transform(lambda x: x.fillna(x.median()))` |
| Remove duplicates | `drop_duplicates(keep='first')` |
| Standardise text | `str.strip().str.title()` |
| Convert type | `pd.to_numeric(errors='coerce')` |
| Parse dates | `pd.to_datetime(infer_datetime_format=True)` |
| Fix impossible values | `loc[(mask), col] = np.nan` |
| Winsorize | `scipy.stats.mstats.winsorize(limits=[0.05,0.05])` |
| Min-Max scaling | `(x - min) / (max - min)` |
| Z-score scaling | `(x - mean) / std` |

**Next:** Week 4 — Exploratory Data Analysis (EDA)